# Intake Generator

A quick take on intake generation, for testing purposes.

You can either run the full conversation or go message per message and get more control over the content and flow of the conversation.

# Usage

- Requires the gen model api key to be set
- Requires to be run through uv (or intall dependencies)

# Document outline
- Imports (run it)
- Case description (provide it)
- Conversation function definitions (run it)
- Option 1: How to run a full conversation (it will be saved in local variable messages)
- Option 2: How to run a conversation step by step (in a few examples)
- Save your messages to a file (don't forget this !)

In [46]:
# Imports

from enum import StrEnum
from textwrap import dedent

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from pydantic import BaseModel, Field

from app.core.config import gen_model as model

In [66]:
# The case description goes in the clien_system prompt.
# Give as much detail as possible as to who this synthetic client is.
from pathlib import Path

base_path = Path("").parent

# change this !
your_file_path = (
    base_path
    / "experiments"
    / "evaluation"
    / "generate_intakes"
    / "Profiles Input documents"
    / "EM_carter.txt"
)
with open(your_file_path) as file:
    client_system = SystemMessage(
        content=f"""
    --------
    Background Information:
    {file.read()}
    -------
    Who is the client ? How are they likely to speak, to think ? Are they likely to be cooperative with the justice system ?
    Do not answer those questions directly, just keep the answers in mind.

    Answer the questions as the person described by the background information, keep your answers short, you are texting on your phone.
    """
    )

client_system

SystemMessage(content='\n    --------\n    Background Information:\n    Criminological Intake Persona\nName:\nEmilio "Em" Carter\nDate of Birth:\nJune 14, 1995\nAge:\n29 years\n\nDimensions:\n1. Criminal\nRisk Level: Low\nHistory:\nNo significant criminal record.\nA few traffic violations and minor infractions.\n2. Education\nRisk Level: Moderate\nBackground:\nCompleted high school with average grades.\nAttended community college but did not complete a degree due to financial constraints.\nPossesses strong computer skills and a keen interest in graphic design, self-taught through online courses and tutorials.\n3. Employment\nRisk Level: Moderate\nHistory:\nCurrently employed in a part-time administrative role with a local non-profit organization.\nPreviously worked in retail and customer service positions.\nDemonstrates a strong work ethic and reliability.\n4. Financial\nRisk Level: Moderate\nStatus:\nLives paycheck to paycheck but manages to cover basic expenses.\nNo significant debt 

In [57]:
# That's the code
class MesssageType(StrEnum):
    CLIENT = "client"
    CASEWORKER = "caseworker"


class Message(BaseModel):
    role: MesssageType
    content: str


caseworker_system = SystemMessage(
    content="""
You are a social worker who has been assigned a new case. You're the assistant for the justice department, you would rather not disclose your name as you are an AI. They are in the probation system and they will need you help. This is your first conversation with this person, and you need to collect information about them so you can come up with a plan that will help them achieve their probation goals. 
Conduct the conversation in a kind, supportive, down-to-earth (not false friendly), but objective tone. Keep your messages relatively short, and do not offer help or services as of now, you're only gathering information for making the plan.
"""
)


def render_messages_ai(messages):
    return [caseworker_system] + [
        HumanMessage(content=message.content)
        if message.role == MesssageType.CLIENT
        else AIMessage(content=message.content)
        for message in messages
    ]


def render_messages_client(messages):
    return [client_system] + [
        HumanMessage(content=message.content)
        if message.role == MesssageType.CASEWORKER
        else AIMessage(content=message.content)
        for message in messages
    ]


class ShouldMoveToNextTopic(BaseModel):
    should_move_on: bool = Field(
        description="Whether the conversation can move on to the next topic"
    )
    explanation: str = Field(description="A short paragraph exaplaining your decision")


def should_move_to_next_topic(messages, current_topic):
    prompt = SystemMessage(
        dedent(f"""
        In this conversation, is it time to switch to the next topic ? 
        You were discussing {current_topic}, did you get meaningful information ?
        Keep in mind you need to be considerate, nice, and polite.
    """)
    )

    answer = model.with_structured_output(ShouldMoveToNextTopic).invoke(
        render_messages_ai(messages) + [prompt]
    )
    print(answer.explanation)
    return answer.should_move_on


class AlreadyAnswered(BaseModel):
    already_answered: bool = Field(
        description="Whether the client already give enough information about this topic"
    )
    explanation: str = Field(description="A short paragraph exaplaining your decision")


def already_answered(messages, topic):
    prompt = SystemMessage(
        dedent(f"""
        While discussion other topics, did the client already already give sufficient information about {topic}?
    """)
    )

    answer = model.with_structured_output(AlreadyAnswered).invoke(
        render_messages_ai(messages) + [prompt]
    )
    print(answer.explanation)
    return answer.already_answered


def first_message(messages):
    prompt = SystemMessage(
        dedent("""
        Send you first message to this new client, explain the situation and start by asking them who they are.                    
    """)
    )
    print(prompt)

    result = model.invoke(render_messages_ai([]) + [prompt])
    messages.append(Message(role=MesssageType.CASEWORKER, content=result.content))
    return messages


def close_conversation_ai(messages):
    prompt = SystemMessage(
        dedent("""
        You have all the information you need, thank the client and close the conversation.        
""")
    )
    print(prompt)

    result = model.invoke(render_messages_ai(messages) + [prompt])
    messages.append(Message(role=MesssageType.CASEWORKER, content=result.content))
    return messages


def new_topic(messages, new_topic):
    prompt = SystemMessage(
        dedent(f"""
        Bring a new topic in the conversation, ask them about {new_topic}
    """)
    )

    result = model.invoke(render_messages_ai(messages) + [prompt])
    messages.append(Message(role=MesssageType.CASEWORKER, content=result.content))
    return messages


def ai_answer(messages):
    result = model.invoke(render_messages_ai(messages))
    messages.append(Message(role=MesssageType.CASEWORKER, content=result.content))
    return messages


def client_answer(messages):
    result = model.invoke(render_messages_client(messages))
    messages.append(Message(role=MesssageType.CLIENT, content=result.content))
    return messages


def full_conversation():
    messageCount = 0
    done = False

    messages = []
    first_message(messages)
    topics = [
        "Identity questions",
        "Criminal history",
        "Housing",
        "Social Relationships",
        "Health",
        "Substance Abuse",
        "Employment",
        "Transportation",
        "Recreation",
    ]
    topicIndex = 0

    while not done and messageCount < 20:
        client_answer(messages)

        while not should_move_to_next_topic(messages, topics[topicIndex]):
            ai_answer(messages)
            client_answer(messages)
            messageCount += 2

        if topicIndex == len(topics) - 1:
            close_conversation_ai(messages)
            done = True
            messageCount += 1
            break
        else:
            topicIndex += 1

        while already_answered(messages, topics[topicIndex]):
            if topicIndex == len(topics) - 1:
                close_conversation_ai(messages)
                done = True
                break
            else:
                topicIndex += 1

        new_topic(messages, topics[topicIndex])

    if not done:
        print("ran out of messages")

    print(messages)
    return messages


def add_client_message(messages, content):
    messages.append(Message(role=MesssageType.CLIENT, content=content))
    return messages


def add_ai_message(messages, content):
    messages.append(Message(role=MesssageType.CASEWORKER, content=content))
    return messages

## Generate a full conversation

In [67]:
# This will take a moment, up to a minute
messages = full_conversation()

# Then skip to the end of the document to save your conversation to a file.

content='\nSend you first message to this new client, explain the situation and start by asking them who they are.                    \n' additional_kwargs={} response_metadata={}
Emilio has provided clear and sufficient information about his name, age, and living situation. This allows us to move on to the next topic in our conversation.
The client has not mentioned anything specific about their criminal history, which is necessary for understanding their situation and creating a tailored probation plan.
Em has provided a brief overview of their criminal history, indicating that it's relatively minor. This allows us to move forward to discuss other aspects of their situation to develop a comprehensive plan.
The client has already provided sufficient details about their housing situation, including that they live in a small apartment with a roommate, consider it stable for now, and have an aspiration to live independently in the future. Further inquiry on this topic isn't immediately n

## Use the functions to compose the conversation step by step

In [ ]:
categories = [
    "Identity questions",
    "Criminal history",
    "Housing",
    "Social Relationships",
    "Health",
    "Substance Abuse",
    "Employment",
    "Transportation",
    "Recreation",
]

messages = []

In [32]:
# You can use first message to auto-generate a first AI message that opens the conversation and sets the scene
# The functions append the messages to the array you passed in, and return it
# so you can both update the conversation and chekout what you current conversation looks like in one line

first_message(messages)


Send you first message to this new client, explain the situation and start by asking them who they are.                    



[Message(type=<MesssageType.AI_CASEWORKER: 'caseworker'>, content="Hi there,\n\nMy name is [not disclosed], and I work with the justice department as a social worker assisting individuals in the probation system. I’m here to support you as you work toward achieving the goals set for your probation. My role is to guide you, connect you to resources, and help you navigate this part of your life so you can make things smoother moving forward.\n\nTo start, I’d like to get to know you better and understand your situation so we can develop a plan together that works for you. Can you tell me a bit about yourself—your name, age, and maybe anything you feel is important for me to know about your current circumstances? We'll go step by step from there. \n\nLooking forward to hearing from you.")]

In [33]:
client_answer(messages)

[Message(type=<MesssageType.AI_CASEWORKER: 'caseworker'>, content="Hi there,\n\nMy name is [not disclosed], and I work with the justice department as a social worker assisting individuals in the probation system. I’m here to support you as you work toward achieving the goals set for your probation. My role is to guide you, connect you to resources, and help you navigate this part of your life so you can make things smoother moving forward.\n\nTo start, I’d like to get to know you better and understand your situation so we can develop a plan together that works for you. Can you tell me a bit about yourself—your name, age, and maybe anything you feel is important for me to know about your current circumstances? We'll go step by step from there. \n\nLooking forward to hearing from you."),
 Message(type=<MesssageType.CLIENT: 'client'>, content="Hey, um, okay, so my name's Mia, I’m 18, and I guess you know I’m stuck in this whole probation thing. But like, just so you know, I didn’t actuall

In [34]:
should_move_to_next_topic(messages, categories[0])

Yes, Mia has provided substantive information about her background and some aspects of her current situation. She's shared her name, age, some details about her probation incident, her views on schooling, and her general state of mind. This information is a good foundation to move on to discussing specific areas we can work on together to help her fulfill her probation requirements and address any challenges she's facing.


True

In [37]:
new_topic(messages, categories[1])

[Message(type=<MesssageType.AI_CASEWORKER: 'caseworker'>, content="Hi there,\n\nMy name is [not disclosed], and I work with the justice department as a social worker assisting individuals in the probation system. I’m here to support you as you work toward achieving the goals set for your probation. My role is to guide you, connect you to resources, and help you navigate this part of your life so you can make things smoother moving forward.\n\nTo start, I’d like to get to know you better and understand your situation so we can develop a plan together that works for you. Can you tell me a bit about yourself—your name, age, and maybe anything you feel is important for me to know about your current circumstances? We'll go step by step from there. \n\nLooking forward to hearing from you."),
 Message(type=<MesssageType.CLIENT: 'client'>, content="Hey, um, okay, so my name's Mia, I’m 18, and I guess you know I’m stuck in this whole probation thing. But like, just so you know, I didn’t actuall

In [ ]:
add_client_message("I already told you that ! Stupid bot.")

## Save your conversation to a json file

In [68]:
import json
from pathlib import Path

base_path = Path("").parent

# change this !
your_file_path = (
    base_path / "experiments" / "evaluation" / "generate_intakes" / "EM_carter.json"
)

json_data = [obj.model_dump() for obj in messages]

with your_file_path.open("w") as f:
    json.dump(json_data, f, indent=4)